<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Find what the bot cannot handle

**[Find what the bot cannot handle](https://learning.nextia-ai.com/courses/text-routing/m10/find-what-the-bot-cannot-handle/)** · Text Classification and Routing: Send Each Message to the Right Place · Case study 2 (Module 10) · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** detect requests that a virtual assistant cannot handle, and hand them to a person. You compare three ways (a confidence threshold, the distance to the nearest example, and an out-of-scope class), measure in-scope accuracy and out-of-scope recall together, and choose the threshold from what a wrong answer and a hand-off cost.

| | |
|---|---|
| **Time** | About 45 minutes in the notebook (the lesson page has the full case study) |
| **Needs** | An internet connection for the setup cells (about 2.5 MB of data, and a 470 MB model the first time). No account and no key. |
| **You should know** | Thresholds and costs, from [The cost of each mistake](https://learning.nextia-ai.com/courses/text-routing/m01/the-cost-of-each-mistake/). Embeddings, from [Sentence embeddings for classification](https://learning.nextia-ai.com/courses/text-routing/m04/sentence-embeddings-for-classification/). Calibration and zones, from [Is the confidence true?](https://learning.nextia-ai.com/courses/text-routing/m08/is-the-confidence-true/) and [Decide, review or send to a person](https://learning.nextia-ai.com/courses/text-routing/m08/decide-review-or-send-to-a-person/). |
| **Data** | CLINC150 by Larson et al. (Clinc), https://github.com/clinc/oos-eval, CC BY 3.0. Real requests written by crowd workers. |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1, sentence-transformers 6.2.0 and torch 2.15.0 (CPU), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m10/find-what-the-bot-cannot-handle/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M10-L02-find-what-the-bot-cannot-handle/find-what-the-bot-cannot-handle-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, sentence-transformers 6.2.0, torch 2.15.0, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2 sentence-transformers==6.2.0 torch==2.15.0
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import sentence_transformers; print("sentence-transformers", sentence_transformers.__version__)
import torch; print("torch", torch.__version__)

## Setup: get the data

Run the next cell. It downloads two files of CLINC150 (about 2.5 MB) from the dataset's own repository at a fixed version, the course's helper `routing.py`, and the recorded answers of a hosted LLM, into a folder `clinc-oos`. It checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 15100 train, 3100 validation and 5500 test requests.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `clinc-oos` (in Colab: the **Files** panel on the left), then run the cell again.

CLINC150 is real data: short requests that crowd workers wrote for an imagined virtual assistant. It has no personal data. "An Evaluation Dataset for Intent Classification and Out-of-Scope Prediction" by Larson et al. (Clinc), https://github.com/clinc/oos-eval, licensed under CC BY 3.0.

In [ ]:
# Setup: download the data into clinc-oos/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

FILES = {
    "data/data_full.json": [
        "https://raw.githubusercontent.com/clinc/oos-eval/828f8093932c8fe6ca7936c3d2e52903b1c523de/data/data_full.json",
        "36923c3705a59e08fe9c3883d8bc2dd966ef93e22cb78ac41171782a698d56e0"
    ],
    "data/domains.json": [
        "https://raw.githubusercontent.com/clinc/oos-eval/828f8093932c8fe6ca7936c3d2e52903b1c523de/data/domains.json",
        "b947b579d3b8e74b06f93b01083d8efaff2888b43a3e362533bd88a6e1211b3a"
    ],
    "routing.py": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/routing.py",
        "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c"
    ],
    "recordings/llm-zero-shot.jsonl": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/M10-L02-find-what-the-bot-cannot-handle/recordings/llm-zero-shot.jsonl",
        "0370d467c09aeaa621a06037c1a5e3cdc783d97b6e01458accaad304eeeb8ebc"
    ]
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("clinc-oos").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder clinc-oos and run again.")

os.chdir(PROJECT)
import json
raw = json.load(open("data/data_full.json", encoding="utf-8"))
print(f"Ready: {len(raw['train']) + len(raw['oos_train'])} train, {len(raw['val']) + len(raw['oos_val'])} validation "
      f"and {len(raw['test']) + len(raw['oos_test'])} test requests.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your values with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the course helper, and a check that does not show the answer.
import time, warnings
import numpy as np
import pandas as pd
import torch
import routing as rt
warnings.filterwarnings("ignore")
torch.manual_seed(36)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data

The next cell reads the three parts of CLINC150 into tables. Each request has a text and an intent: one of 150 things the assistant can do, or `oos` (out of scope) when it is none of them.

> **Check.** You should see 15,100 train, 3,100 validation and 5,500 test requests. The share of out-of-scope requests is very different in each part: 0.7%, 3.2% and 18.2%.

In [ ]:
import json
raw = json.load(open("data/data_full.json", encoding="utf-8"))
DOMAIN_OF = {i: d for d, intents in json.load(open("data/domains.json")).items() for i in intents}

def part(*keys):
    """One part of CLINC150 as a table. The out-of-scope requests have the intent "oos"."""
    rows = [(text, intent) for k in keys for text, intent in raw[k]]
    df = pd.DataFrame(rows, columns=["text", "intent"])
    df["domain"] = df.intent.map(DOMAIN_OF).fillna("out of scope")
    return df

train, validation, test = part("train", "oos_train"), part("val", "oos_val"), part("test", "oos_test")
# CLINC150 has no request IDs: we number the rows in the order of the file.
for prefix, df in [("TR", train), ("VA", validation), ("TE", test)]:
    df.insert(0, "message_id", [f"C-{prefix}-{i:05d}" for i in range(1, len(df) + 1)])
for name, df in [("train", train), ("validation", validation), ("test", test)]:
    oos = (df.intent == "oos").sum()
    print(f"{name:10}: {len(df):6,} requests, {df.intent.nunique() - 1} intents, out of scope {oos:5,} ({oos / len(df):.1%})")
print("domains:", sorted(set(DOMAIN_OF.values())))

The next cell makes the first checks: length, gaps, repeats and the number of requests per intent. It also prints three out-of-scope requests.

> **Check.** You should see a median of 8 words, no gaps, and exactly 100, 20 and 30 requests per intent. The first out-of-scope request is about a bank's overdraft fee, although the assistant has 15 banking intents.

In [ ]:
words = train.text.str.split().str.len()
print(f"words per request: median {words.median():.0f}, 90% have at most {words.quantile(0.9):.0f}, longest {words.max()}")
print("missing or empty texts:", int((train.text.str.strip() == "").sum() + train.text.isna().sum()))
print("repeated requests in train:", int(train.text.duplicated().sum()),
      "| test requests also in train:", int(test.text.isin(train.text).sum()))
print("in-scope requests per intent: train", train[train.intent != "oos"].intent.value_counts().unique(),
      "| validation", validation[validation.intent != "oos"].intent.value_counts().unique(),
      "| test", test[test.intent != "oos"].intent.value_counts().unique())
print("\nThree out-of-scope requests from the training part:")
for t in train[train.intent == "oos"].text.head(3):
    print("  -", t)

## 2. Two outcomes, two costs

The bot can do two things with a request: **answer** with an intent, or **hand off** ("I cannot help with that, here is a person"). The next cell sets the costs (assumptions of this case study):

- a wrong answer: US$4, for a wrong intent **or** for an out-of-scope request that the bot answers anyway;
- an in-scope request handed to a person: US$1.50, one minute of an agent that the bot could have saved;
- an out-of-scope request handed to a person: US$0, because that is the right place for it.

The hand-off uses the name `needs_person`, as in Larkfield's inbox, so the course's `routing.py` computes the cost unchanged. We assume that 1 request in 5 is out of scope (`SHARE_OOS`), and every cost per 1,000 requests uses that share.

> **Check.** Everything to a person costs 1,200 dollars per 1,000 requests: 800 in-scope requests at US$1.50.

In [ ]:
# The case study's helpers: the course's cost rule (routing.py), with the bot's two outcomes.
from sklearn.metrics import f1_score
HAND_OFF = "needs_person"   # the bot says "I cannot help with that, here is a person"
COSTS = dict(rt.COSTS,
             team_message_sent_to_person=1.50,  # an in-scope request handed to a person: one minute of an agent
             wrong_queue=4.0,                   # a wrong automatic answer to an in-scope request
             needs_person_sent_to_team=4.0)     # an out-of-scope request answered as if it were an intent
SHARE_OOS = 0.20            # assumption: 1 request in 5 that reaches the bot is out of scope
CPU_HOUR = 0.05             # dollars per CPU-hour (the course's assumption)
INTENTS = sorted(train.intent[train.intent != "oos"].unique())

def truth_of(part):
    """The columns that routing.py needs. An out-of-scope request's right answer is a person."""
    return pd.DataFrame({"route": part.intent.replace("oos", HAND_OFF).values, "topic": part.intent.values,
                         "priority": "normal", "language": "en"}, index=part.index)

def decide(intents, confidence, threshold):
    """Answer with the intent when the confidence is at least the threshold; otherwise hand off.
    An intent called "oos" (a model that learned the out-of-scope class) is a hand-off too."""
    intents = np.asarray(intents, dtype=object)
    sure = (np.asarray(confidence) >= threshold) & (intents != "oos")
    return np.where(sure, intents, HAND_OFF)

def score(name, part, decisions, run_per_1000=0.0, ms=None, share=SHARE_OOS):
    """One row of the comparison table. The cost per 1,000 requests is for a stream with `share` out of scope."""
    truth = truth_of(part)
    d = pd.Series(np.asarray(decisions, dtype=object), index=part.index)
    cost = pd.Series([rt.message_cost(t, p, "normal", tp, COSTS) for t, p, tp in
                      zip(truth.route, d, truth.topic)], index=part.index)
    ins, oos = part.intent != "oos", part.intent == "oos"
    mistakes = 1000 * ((1 - share) * cost[ins].mean() + share * cost[oos].mean())
    return {"method": name,
            "in_scope_accuracy": round(float((d[ins] == part.intent[ins]).mean()), 3),
            "oos_recall": round(float((d[oos] == HAND_OFF).mean()), 3),
            "macro_f1": round(float(f1_score(truth.route, d, average="macro", labels=INTENTS + [HAND_OFF])), 3),
            "hand_off_share": round(float((1 - share) * (d[ins] == HAND_OFF).mean() + share * (d[oos] == HAND_OFF).mean()), 3),
            "wrong_answers": round(float((1 - share) * ((d[ins] != HAND_OFF) & (d[ins] != part.intent[ins])).mean()
                                         + share * (d[oos] != HAND_OFF).mean()), 3),
            "mistakes_per_1000": round(mistakes, 1), "run_per_1000": round(run_per_1000, 3),
            "total_per_1000": round(mistakes + run_per_1000, 1),
            "ms_per_message": None if ms is None else round(ms, 2)}

def table(rows):
    """The rows as a table, cheapest total first."""
    return pd.DataFrame(rows).sort_values("total_per_1000").reset_index(drop=True)

def sweep(name, part, intents, confidence, thresholds):
    """Score every threshold: the trade-off between in-scope accuracy and out-of-scope recall."""
    return pd.DataFrame([{"threshold": t, **score(name, part, decide(intents, confidence, t))} for t in thresholds])

def cpu_cost(ms):
    return ms / 1000 / 3600 * CPU_HOUR * 1000   # dollars per 1,000 requests at `ms` milliseconds each

def ms_per_message(predict, texts, n=200):
    t0 = time.perf_counter()
    for t in texts[:n]:
        predict([t])
    return (time.perf_counter() - t0) / n * 1000

print("everything to a person:", score("everything to a person", validation, [HAND_OFF] * len(validation))["total_per_1000"],
      "dollars per 1,000 requests")

## 3. The rule on a tiny example

Six requests, made up on purpose. The model gives each one an intent and a confidence. Below the threshold, the bot hands off.

> **Predict.** At the threshold 0.50, which requests cost money? Is 0.75 cheaper or dearer? Write your answers down, then run the cell.

In [ ]:
# A practice problem we made on purpose: 6 requests, the model's intent and confidence, two thresholds.
tiny = pd.DataFrame({
    "text": ["what's my checking balance", "book a table for two at 8", "how do i fix my sink",
             "transfer 50 dollars to sam", "what is the tallest mountain", "set an alarm for 6"],
    "intent": ["balance", "restaurant_reservation", "oos", "transfer", "oos", "alarm"],
    "model_says": ["balance", "restaurant_reservation", "schedule_maintenance", "transfer", "distance", "reminder"],
    "confidence": [0.91, 0.62, 0.34, 0.48, 0.71, 0.55]})
for threshold in [0.50, 0.75]:
    tiny[f"decision_{threshold}"] = decide(tiny.model_says, tiny.confidence, threshold)
    tiny[f"cost_{threshold}"] = [rt.message_cost(t, d, "normal", i, COSTS) for t, d, i in
                                 zip(tiny.intent.replace("oos", HAND_OFF), tiny[f"decision_{threshold}"], tiny.intent)]
print(tiny[["cost_0.5", "cost_0.75"]].sum().to_dict())
tiny.drop(columns="text")

> **Check.** You should see a total of 9.5 dollars at 0.50 and 4.5 at 0.75. At 0.50, the bot answers the out-of-scope "tallest mountain" request (0.71) and gets the alarm wrong (0.55). At 0.75 it hands both off, and also two in-scope requests that it would have got right.

## 4. The baseline: TF-IDF that always answers

The next cell trains the course's TF-IDF + logistic regression on the 15,000 in-scope training requests only. It knows 150 intents and has no way to say "none of them". It takes about 10 seconds.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, make_pipeline

def tfidf_lr():
    """The course project's TF-IDF + logistic regression: words and word pairs, plus letter groups."""
    words = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")
    letters = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)
    return make_pipeline(FeatureUnion([("words", words), ("letters", letters)]), LogisticRegression(C=10, max_iter=3000))

in_scope = train[train.intent != "oos"]
t0 = time.perf_counter()
tfidf = tfidf_lr().fit(in_scope.text, in_scope.intent)        # learns the 150 intents only
print(f"fitted on {len(in_scope):,} in-scope requests in {time.perf_counter() - t0:.0f} s")
p = tfidf.predict_proba(validation.text)
tfidf_val = (tfidf.classes_[p.argmax(axis=1)], p.max(axis=1))   # (intent, confidence) for each request
tfidf_ms = ms_per_message(tfidf.predict_proba, list(validation.text))
rows_val = [score("everything to a person", validation, [HAND_OFF] * len(validation)),
            score("TF-IDF + LR, always answers", validation, tfidf_val[0], cpu_cost(tfidf_ms), tfidf_ms)]
table(rows_val)

> **Check.** In-scope accuracy is 0.924, out-of-scope recall is 0, and the cost is 1043.2 dollars per 1,000. A model that always answers gets every out-of-scope request wrong.

## 5. Is the confidence true?

Before we use the confidence as a rule, we check it, as in Module 8. The next cell prints the reliability table of the in-scope validation requests, and the median confidence of in-scope and out-of-scope requests.

In [ ]:
# Is the confidence true? The reliability table of the in-scope validation requests (Module 8).
ins = (validation.intent != "oos").to_numpy()
rel, ece = rt.reliability(tfidf_val[1][ins], tfidf_val[0][ins] == validation.intent[ins])
print("expected calibration error:", ece)
print("median confidence: in scope", round(float(np.median(tfidf_val[1][ins])), 3),
      "| out of scope", round(float(np.median(tfidf_val[1][~ins])), 3))
rel

> **Check.** The expected calibration error is about 0.0967. The model is too shy: where it says 0.45, it is right about 70% of the time. Out-of-scope requests get a much lower confidence (median about 0.11) than in-scope ones (about 0.94). That gap is what a threshold uses.

## 6. Choose the threshold from the costs

The next cell tries every threshold from 0.00 to 0.99 on the validation requests and keeps the cheapest. A table shows a few thresholds, and a chart shows the cost of each.

> **Predict.** If the confidence were true, the break-even point would be 1 − 1.50 ÷ 4 = 0.625. Will the cheapest threshold be above or below 0.625 for this shy model?

In [ ]:
thresholds = np.round(np.arange(0.0, 1.0, 0.01), 2)
curve_tfidf = sweep("TF-IDF + LR", validation, *tfidf_val, thresholds)
best = curve_tfidf.loc[curve_tfidf.total_per_1000.idxmin()]
T_TFIDF = float(best.threshold)
print(f"lowest cost on validation: hand off below {T_TFIDF:.2f}; in-scope accuracy {best.in_scope_accuracy:.3f}, "
      f"out-of-scope recall {best.oos_recall:.3f}, {best.total_per_1000:.1f} dollars per 1,000")
shown = curve_tfidf.threshold.isin([0.0, 0.1, 0.2, T_TFIDF, 0.4, 0.5, 0.6, 0.7, 0.9])
curve_tfidf.loc[shown, ["threshold", "in_scope_accuracy", "oos_recall", "hand_off_share", "wrong_answers", "total_per_1000"]]

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(curve_tfidf.threshold, curve_tfidf.total_per_1000)
ax.axvline(T_TFIDF, ls="--", color="grey")
ax.set_xlabel("hand off below this confidence"); ax.set_ylabel("dollars per 1,000 requests")
ax.set_title("TF-IDF + LR on the validation requests")
plt.show()

> **Check.** The cheapest threshold is 0.55: the bot hands off when its confidence is below 0.55.

## 7. Two more ways, with embeddings

The next cell turns every request into an embedding with the course's older embedding model, and trains a logistic regression on the in-scope ones. It downloads the model (about 470 MB) the first time. It takes about 1 minute on a recent laptop, and longer on free Colab.

In [ ]:
from sentence_transformers import SentenceTransformer
torch.set_num_threads(4)
# The course's older embedding model, frozen. Each request becomes 384 numbers of length 1.
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                               revision="e8f8c211226b894fcb81acc59f3b34ba3efd5f42", device="cpu")
t0 = time.perf_counter()
E = {name: embedder.encode(list(df.text), batch_size=128, normalize_embeddings=True)
     for name, df in [("train", train), ("validation", validation), ("test", test)]}
print(f"{sum(len(v) for v in E.values()):,} requests embedded in {time.perf_counter() - t0:.0f} s")
ins_train = (train.intent != "oos").to_numpy()
frozen = LogisticRegression(C=10, max_iter=3000).fit(E["train"][ins_train], train.intent[ins_train])
p = frozen.predict_proba(E["validation"])
frozen_val = (frozen.classes_[p.argmax(axis=1)], p.max(axis=1))
frozen_ms = ms_per_message(lambda t: frozen.predict_proba(embedder.encode(t, normalize_embeddings=True)),
                           list(validation.text), n=100)
curve_frozen = sweep("embeddings + LR", validation, *frozen_val, thresholds)
T_FROZEN = float(curve_frozen.loc[curve_frozen.total_per_1000.idxmin(), "threshold"])
print("embeddings + LR: lowest cost on validation when it hands off below", T_FROZEN)

The second way does not need a classifier at all. For each request, it finds the **most similar in-scope training request** (cosine similarity of the embeddings). If that request is close enough, the bot answers with its intent. If not, it hands off.

> **Check.** In-scope requests have a median similarity of about 0.88 to their nearest example; out-of-scope requests about 0.53.

In [ ]:
# Distance to the nearest example: how similar is the request to the closest in-scope training request?
bank, bank_intent = E["train"][ins_train], train.intent[ins_train].to_numpy()

def nearest(X, size=1000):
    """For each row of X: the intent of the most similar in-scope training request, and the cosine similarity."""
    intents, sims = [], []
    for i in range(0, len(X), size):
        S = X[i:i + size] @ bank.T          # the vectors have length 1, so this is the cosine similarity
        intents.append(bank_intent[S.argmax(axis=1)])
        sims.append(S.max(axis=1))
    return np.concatenate(intents), np.concatenate(sims)

nearest_val = nearest(E["validation"])
curve_nearest = sweep("nearest example", validation, *nearest_val, thresholds)
T_NEAREST = float(curve_nearest.loc[curve_nearest.total_per_1000.idxmin(), "threshold"])
ins = (validation.intent != "oos").to_numpy()
print("median similarity to the nearest example: in scope", round(float(np.median(nearest_val[1][ins])), 3),
      "| out of scope", round(float(np.median(nearest_val[1][~ins])), 3))
print("nearest example: lowest cost on validation when it hands off below", T_NEAREST)

## 8. An explicit out-of-scope class

The third way: train the classifier with the 100 out-of-scope training requests as a 151st intent, `oos`. When the model predicts `oos`, the bot hands off. We try it with no threshold, and with a threshold chosen on validation.

In [ ]:
# An explicit class: the 100 out-of-scope training requests become a 151st intent, "oos".
oos_tfidf = tfidf_lr().fit(train.text, train.intent)
p = oos_tfidf.predict_proba(validation.text)
oos_tfidf_val = (oos_tfidf.classes_[p.argmax(axis=1)], p.max(axis=1))
oos_frozen = LogisticRegression(C=10, max_iter=3000).fit(E["train"], train.intent)
p = oos_frozen.predict_proba(E["validation"])
oos_frozen_val = (oos_frozen.classes_[p.argmax(axis=1)], p.max(axis=1))
curve_oos_tfidf = sweep("TF-IDF + LR, oos class", validation, *oos_tfidf_val, thresholds)
curve_oos_frozen = sweep("embeddings + LR, oos class", validation, *oos_frozen_val, thresholds)
T_OOS_TFIDF = float(curve_oos_tfidf.loc[curve_oos_tfidf.total_per_1000.idxmin(), "threshold"])
T_OOS_FROZEN = float(curve_oos_frozen.loc[curve_oos_frozen.total_per_1000.idxmin(), "threshold"])
rows_val += [score("TF-IDF + LR, oos class, no threshold", validation, decide(*oos_tfidf_val, 0.0),
                   cpu_cost(tfidf_ms), tfidf_ms),
             score("embeddings + LR, oos class, no threshold", validation, decide(*oos_frozen_val, 0.0),
                   cpu_cost(frozen_ms), frozen_ms)]
print("thresholds with the oos class: TF-IDF", T_OOS_TFIDF, "| embeddings", T_OOS_FROZEN)

The next cell puts every method at its own cheapest threshold into one table, cheapest first.

> **Check.** The cheapest row is embeddings + LR with the oos class, handing off below 0.62, at 212.8 dollars per 1,000. With no threshold, the oos class finds only about half of the out-of-scope requests: 100 examples cannot show everything that is out of scope.

In [ ]:
# Every method at the threshold chosen on validation (the lowest cost), one row each.
CHOSEN = {"TF-IDF + LR": (tfidf_val, T_TFIDF, tfidf_ms), "embeddings + LR": (frozen_val, T_FROZEN, frozen_ms),
          "nearest example": (nearest_val, T_NEAREST, frozen_ms),
          "TF-IDF + LR, oos class": (oos_tfidf_val, T_OOS_TFIDF, tfidf_ms),
          "embeddings + LR, oos class": (oos_frozen_val, T_OOS_FROZEN, frozen_ms)}
rows_val += [score(f"{m}, hand off below {t:.2f}", validation, decide(*v, t), cpu_cost(ms), ms)
             for m, (v, t, ms) in CHOSEN.items()]
table(rows_val)

## 9. The hosted LLM, replayed

A hosted LLM (chat-small) got the 150 intents by domain and the choice `out_of_scope`. Its answers were recorded once for a sample, so you need no key. The next cells make the samples, show the prompt, and score the answers next to the other methods on the same sample.

In [ ]:
# The hosted LLM's answers were recorded once for a sample with the same share of out-of-scope requests as
# the test part (18.2%): 3 per intent + 100 out of scope (validation), 6 per intent + 200 (test).
def sample(df, per_intent, n_oos):
    shuffled = df.sample(frac=1, random_state=36)
    return pd.concat([shuffled[shuffled.intent != "oos"].groupby("intent").head(per_intent),
                      shuffled[shuffled.intent == "oos"].head(n_oos)]).sort_index()

validation_sample, test_sample = sample(validation, 3, 100), sample(test, 6, 200)
print(f"samples with recorded LLM answers: validation {len(validation_sample)}, test {len(test_sample)}")

In [ ]:
# The prompt of the recorded answers: the 150 intents by domain, and a way out.
by_domain = "\n".join(f"- {d}: " + ", ".join(sorted(i for i in INTENTS if DOMAIN_OF[i] == d))
                       for d in sorted(set(DOMAIN_OF.values())))
SYSTEM = ("You are the intent classifier of a virtual assistant. Choose exactly one intent from this list "
          f"(grouped by domain):\n{by_domain}\n"
          "If the request fits none of these intents, choose out_of_scope. "
          "Answer in JSON with the intent and your confidence from 0 to 1.")
print(SYSTEM[:400], "...")
print(f"\n{len(SYSTEM.split())} words in the instructions")

In [ ]:
def recorded(mode):
    """The recorded answers of the hosted LLM (chat-small), one line per request."""
    rows = [json.loads(x) for x in open(f"recordings/llm-{mode}.jsonl", encoding="utf-8")]
    return pd.DataFrame(rows).set_index("message_id")

zero = recorded("zero-shot")
filtered = zero.get("filtered", pd.Series(False, index=zero.index)).fillna(False).astype(bool)
answered = zero[~filtered]          # the provider's content filter refused a few requests: no answer, a hand-off
PRICE_IN, PRICE_OUT = 0.10, 0.50   # chat-small, dollars per 1 million tokens (the course's price table)
llm_run = (answered.tokens_in.mean() * PRICE_IN + answered.tokens_out.mean() * PRICE_OUT) / 1e6 * 1000
llm_ms = answered.latency_s.median() * 1000
print(f"zero-shot: {len(zero)} requests, {len(answered)} answers, all valid: {answered.valid.all()}, "
      f"refused by the content filter: {filtered.sum()} | tokens in {answered.tokens_in.mean():.0f}, "
      f"out {answered.tokens_out.mean():.0f} | {llm_run:.3f} dollars per 1,000 | median {llm_ms:.0f} ms")

def llm_answers(part):
    a = zero.loc[part.message_id]
    return a.intent.replace("out_of_scope", "oos").to_numpy(), a.confidence.to_numpy()

llm_val = llm_answers(validation_sample)
curve_llm = sweep("LLM zero-shot", validation_sample, *llm_val, thresholds)
T_LLM = float(curve_llm.loc[curve_llm.total_per_1000.idxmin(), "threshold"])
vs = validation_sample.index
rows_sample = [score(f"{m}, hand off below {t:.2f}", validation_sample,
                     decide(v[0][vs], v[1][vs], t), cpu_cost(ms), ms) for m, (v, t, ms) in CHOSEN.items()]
rows_sample += [score("LLM zero-shot, its own out_of_scope only", validation_sample, decide(*llm_val, 0.0), llm_run, llm_ms),
                score(f"LLM zero-shot, hand off below {T_LLM:.2f}", validation_sample, decide(*llm_val, T_LLM), llm_run, llm_ms)]
table(rows_sample)

## 10. The trade-off as a picture

Each threshold gives one point: out-of-scope recall across, in-scope accuracy up. Moving the threshold up finds more out-of-scope requests and answers fewer in-scope ones. A curve that stays higher is better at every point.

In [ ]:
# The trade-off: each threshold gives one point (out-of-scope recall, in-scope accuracy). Up and right is better.
curves = {"TF-IDF + LR": curve_tfidf, "embeddings + LR": curve_frozen, "nearest example": curve_nearest,
          "TF-IDF + LR, oos class": curve_oos_tfidf, "embeddings + LR, oos class": curve_oos_frozen}
for m, c in curves.items():
    at90 = c[c.oos_recall >= 0.9].in_scope_accuracy.max()
    print(f"{m:28} in-scope accuracy when it finds at least 90% of out-of-scope requests: {at90:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for m, c in curves.items():
    ax.plot(c.oos_recall, c.in_scope_accuracy, label=m)
ax.set_xlabel("out-of-scope recall (handed to a person)"); ax.set_ylabel("in-scope accuracy (answered right)")
ax.set_xlim(0, 1); ax.set_ylim(0.5, 1); ax.legend(); ax.set_title("Validation: one point per threshold")
plt.show()

## 11. Change one thing: the share of out-of-scope requests

The share of out-of-scope requests is an assumption. The next cell chooses the threshold of the best method again for 5%, 20% and 40%.

> **Predict.** With more out-of-scope requests, does the threshold go up or down?

In [ ]:
# Change one thing: the share of out-of-scope requests. Choose the threshold again for each share.
for share in [0.05, 0.20, 0.40]:
    c = pd.DataFrame([{"threshold": t, **score("", validation, decide(*oos_frozen_val, t), share=share)}
                      for t in thresholds])
    b = c.loc[c.total_per_1000.idxmin()]
    print(f"{share:.0%} out of scope: hand off below {b.threshold:.2f}, out-of-scope recall {b.oos_recall:.3f}, "
          f"in-scope accuracy {b.in_scope_accuracy:.3f}, {b.total_per_1000:.1f} dollars per 1,000")

> **Check.** The threshold goes up with the share: more out-of-scope requests make each answer riskier.

## 12. Your turn: a wrong answer costs more

> **Your turn.** In a bank, a wrong answer can move money. Set the cost of a wrong answer to 10 dollars and find the cheapest threshold for embeddings + LR with the oos class. Use `sweep()` with `oos_frozen_val` and `thresholds`. Put the threshold in `threshold_10`.

In [ ]:
# Change one thing: a wrong answer costs 10 dollars, not 4 (a bank, where a wrong answer moves money).
COSTS_4 = dict(COSTS)
COSTS.update(wrong_queue=10.0, needs_person_sent_to_team=10.0)
threshold_10 = ...   # the threshold with the lowest cost for embeddings + LR with the oos class (use sweep)
COSTS.update(COSTS_4)
if threshold_10 is not ...:
    print("a wrong answer costs 10 dollars: hand off below", threshold_10)

In [ ]:
expect('threshold_10', threshold_10, 0.69, tolerance=1e-09)

## 13. The test requests, once

Every method and threshold was chosen on validation. The next cell scores them on the 5,500 test requests, once. Nothing changes after this cell.

In [ ]:
# The test requests, once. Every method and threshold was chosen on validation; nothing changes after this cell.
p = tfidf.predict_proba(test.text);       tfidf_test = (tfidf.classes_[p.argmax(axis=1)], p.max(axis=1))
p = frozen.predict_proba(E["test"]);      frozen_test = (frozen.classes_[p.argmax(axis=1)], p.max(axis=1))
p = oos_tfidf.predict_proba(test.text);   oos_tfidf_test = (oos_tfidf.classes_[p.argmax(axis=1)], p.max(axis=1))
p = oos_frozen.predict_proba(E["test"]);  oos_frozen_test = (oos_frozen.classes_[p.argmax(axis=1)], p.max(axis=1))
nearest_test = nearest(E["test"])
TESTS = {"TF-IDF + LR": tfidf_test, "embeddings + LR": frozen_test, "nearest example": nearest_test,
         "TF-IDF + LR, oos class": oos_tfidf_test, "embeddings + LR, oos class": oos_frozen_test}
rows_test = [score("everything to a person", test, [HAND_OFF] * len(test)),
             score("TF-IDF + LR, always answers", test, tfidf_test[0], cpu_cost(tfidf_ms), tfidf_ms)]
rows_test += [score(f"{m}, hand off below {t:.2f}", test, decide(*TESTS[m], t), cpu_cost(ms), ms)
              for m, (v, t, ms) in CHOSEN.items()]
table(rows_test)

The LLM was recorded for a sample of 1,100 test requests. The next cell compares every method on that same sample.

In [ ]:
# The LLM was recorded for a sample of the test requests: compare every method on that same sample.
ts = test_sample.index
llm_test = llm_answers(test_sample)
rows_test_sample = [score(f"{m}, hand off below {t:.2f}", test_sample, decide(TESTS[m][0][ts], TESTS[m][1][ts], t),
                          cpu_cost(ms), ms) for m, (v, t, ms) in CHOSEN.items()]
rows_test_sample += [score("LLM zero-shot, its own out_of_scope only", test_sample, decide(*llm_test, 0.0), llm_run, llm_ms),
                     score(f"LLM zero-shot, hand off below {T_LLM:.2f}", test_sample, decide(*llm_test, T_LLM), llm_run, llm_ms)]
table(rows_test_sample)

The trade-off on the test requests, with a dot at each threshold chosen on validation. No choice is made here.

In [ ]:
# The trade-off on the test requests, for the picture (no choice is made here).
test_curves = {m: sweep(m, test, *TESTS[m], thresholds) for m in CHOSEN}
for m, c in test_curves.items():
    at90 = c[c.oos_recall >= 0.9].in_scope_accuracy.max()
    print(f"{m:28} test: in-scope accuracy when it finds at least 90% of out-of-scope requests: {at90:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for m, c in test_curves.items():
    line, = ax.plot(c.oos_recall, c.in_scope_accuracy, label=m)
    t = CHOSEN[m][1]
    chosen = c[c.threshold == t].iloc[0]
    ax.plot(chosen.oos_recall, chosen.in_scope_accuracy, "o", color=line.get_color())
ax.set_xlabel("out-of-scope recall"); ax.set_ylabel("in-scope accuracy")
ax.set_xlim(0, 1); ax.set_ylim(0.5, 1); ax.legend(); ax.set_title("Test: the dots are the thresholds chosen on validation")
plt.show()

> **Check.** The chosen method, embeddings + LR with the oos class below 0.62, costs 271.5 dollars per 1,000 on test. It answers 0.866 of the in-scope requests right and hands 0.892 of the out-of-scope ones to a person.

## 14. What still goes wrong

The next cell lists the out-of-scope test requests that the bot answers anyway, the most confident first, and counts the in-scope requests that it hands off.

In [ ]:
# Where does the chosen method still go wrong on the test requests?
chosen_intent, chosen_conf = oos_frozen_test
d = decide(chosen_intent, chosen_conf, T_OOS_FROZEN)
oos = (test.intent == "oos").to_numpy()
missed = test[oos & (d != HAND_OFF)].assign(answered_as=d[oos & (d != HAND_OFF)],
                                            confidence=chosen_conf[oos & (d != HAND_OFF)].round(3))
print(f"out-of-scope requests answered by the bot: {len(missed)} of {oos.sum()}")
handed = test[~oos & (d == HAND_OFF)]
print(f"in-scope requests handed to a person: {len(handed)} of {(~oos).sum()}; most often:",
      handed.intent.value_counts().head(5).to_dict())
missed.sort_values("confidence", ascending=False)[["text", "answered_as", "confidence"]].head(8)

> **Your turn.** Which domain's intents answer the most out-of-scope test requests? Map each `answered_as` intent of `missed` to its domain with `DOMAIN_OF`, count, and put the top domain in `top_domain`.

In [ ]:
# Which domains answer the out-of-scope requests that slip through? Map each answered intent to its domain (DOMAIN_OF).
top_domain = ...   # the domain whose intents answered the most out-of-scope test requests

In [ ]:
expect('top_domain', top_domain, 'utility')

## 15. New requests, end to end

The next cell runs the chosen method on eight requests that we wrote for this case study, from the text to the decision.

> **Predict.** Which of the eight will the bot hand off? "What is the meaning of life" is one of the 150 intents (small talk).

In [ ]:
# New input: requests that we wrote for this case study (not from CLINC150).
new = pd.DataFrame({"text": [
    "how much money is in my savings account",
    "please move 200 dollars from checking to savings",
    "what's the weather going to be like in boston tomorrow",
    "can you recommend a good plumber near me",
    "my card was stolen, someone is using it right now",
    "what is the meaning of life",
    "how many calories are in a banana",
    "write a poem about my cat"]})
X = embedder.encode(list(new.text), normalize_embeddings=True)
p = oos_frozen.predict_proba(X)
new["intent"], new["confidence"] = oos_frozen.classes_[p.argmax(axis=1)], p.max(axis=1).round(3)
new["decision"] = decide(new.intent, new.confidence, T_OOS_FROZEN)
new["decision"] = np.where(new.decision == HAND_OFF, "hand off to a person", "answer: " + new.decision)
new

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m10/find-what-the-bot-cannot-handle/) has the full case study: the business question, the pictures, the recommendation, responsible use, how to make it a portfolio piece, and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Sort requests with few labels](https://learning.nextia-ai.com/courses/text-routing/m10/sort-requests-with-few-labels/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m10/find-what-the-bot-cannot-handle/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). CLINC150 by Larson et al. (Clinc), CC BY 3.0.